# HWO target evidence cross-match -- Colab walkthrough

**Purpose**: cross-match the 164-target Habitable Worlds Observatory Direct Imaging ExEP Mission Star List (Mamajek & Stapelfeldt 2024) against the Hipparcos-Gaia Catalog of Accelerations (Brandt 2021) and report per-category evidence completeness.

**Data terms**: both sources are public, no credentials required. Live queries to the NASA Exoplanet Archive and VizieR (CDS).

**Runtime estimate**: under a minute on Colab's default runtime.

This notebook makes no claim that any specific star does or does not host a planet or companion beyond what the cited catalogs directly report -- see `docs/METHODS.md` in the repository.

In [ ]:
%pip install -q git+https://github.com/Biswajit1999/hwo-target-evidence-atlas.git

import hwoevidence
print("hwoevidence", hwoevidence.__version__)

In [ ]:
from pathlib import Path
from hwoevidence.data import CachePaths, load_di_stars_exep, load_hgca_for_hip
from hwoevidence.evidence import build_evidence_matrix
from hwoevidence.conflicts import find_hgca_accelerators_without_archive_listed_planets

paths = CachePaths(cache_dir=Path("data_cache"))
stars = load_di_stars_exep(paths)
print(f"DI_STARS_EXEP: {len(stars)} rows")

hip_numbers = sorted(set(stars["hip_number"].dropna().astype(int).tolist()))
hgca = load_hgca_for_hip(hip_numbers, paths)
print(f"HGCA cross-match: {len(hgca)}/{len(hip_numbers)} unique HIP numbers matched")

evidence = build_evidence_matrix(stars, hgca)
assert len(evidence) == len(stars)
evidence.head()

In [ ]:
import matplotlib.pyplot as plt

# Only the four real observational-coverage categories feed into
# evidence_completeness -- archive_planet_flag_populated and
# archive_disk_flag_populated are tracked separately and excluded because
# they are populated for essentially every target regardless of whether a
# real companion or exozodi observation was performed (see
# docs/LIMITATIONS.md).
cats = ["stellar_parameters", "activity", "multiplicity", "astrometric_acceleration"]
fracs = evidence[cats].mean() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(cats, fracs, color="#2b6cb0")
ax.set_xlim(0, 105)
ax.set_xlabel("Fraction of targets with real observational coverage (%)")
ax.set_title(f"HWO target evidence coverage by category (n={len(evidence)})")
for bar, frac in zip(bars, fracs):
    ax.text(frac + 1, bar.get_y() + bar.get_height() / 2, f"{frac:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## HGCA accelerators without an archive-listed planet

Targets with a statistically significant HGCA astrometric acceleration but no archive-listed planet in `sy_planets_flag` -- candidates for follow-up, **not** an established catalog conflict: `sy_planets_flag == "N"` means no archive-listed planet, not a confirmed single-star determination, so the acceleration may come from a companion the archive simply hasn't listed as a planet.

In [ ]:
accelerators = find_hgca_accelerators_without_archive_listed_planets(evidence)
n_fdr = int(accelerators["hgca_significant_accel_fdr"].sum())
print(f"{len(accelerators)} candidates out of {len(evidence)} targets at raw p<0.01 "
      f"({n_fdr} survive Benjamini-Hochberg FDR correction)")
accelerators.head(10)

## Limitations (summary; full list in `docs/LIMITATIONS.md`)

- Multiplicity evidence is a Washington Double Star (visual binary) cross-match only; it cannot detect close or non-resolved companions.
- The exozodiacal-dust flag is a proxy, not a dedicated survey (only ~23/164 targets have a real HOSTS-survey observation).
- A significant HGCA acceleration is evidence of an unmodeled companion, not a mass/period/nature determination.